# Packages

In [7]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import random as rd

from surprise import AlgoBase
from surprise.prediction_algorithms.predictions import PredictionImpossible
from sklearn.preprocessing import MinMaxScaler

from models import ContentBased
from loaders import load_ratings
from loaders import load_items
from constants import Constant as C

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Explore and select content features

In [2]:
df_items = load_items()
df_ratings = load_ratings()

# Example 1 : create title_length features
df_features = df_items[C.LABEL_COL].apply(lambda x: len(x)).to_frame('n_character_title')
display(df_features.head())

# (explore here other features)

#df_rank = df_items[C.]

df_movie_date = df_items[C.LABEL_COL].str.extract(r'\((\d{4})\)').astype(float)
df_movie_date.columns = ['release_year']

# Attention : s'il manque l'année pour certains films, cela va créer des cases vides (NaN) 
# qui font planter la régression. On remplace les vides par l'année médiane :
df_movie_date['release_year'] = df_movie_date['release_year'].fillna(df_movie_date['release_year'].median())

display(df_movie_date.head())

df_movie_genres = df_items[C.GENRES_COL].str.split('|')
display(df_movie_genres)


,n_character_title
movieId,
1,16
2,14
3,23
4,24
5,34


,release_year
movieId,
1,1995.0
2,1995.0
3,1995.0
4,1995.0
5,1995.0


movieId
1         [Adventure, Animation, Children, Comedy, Fantasy]
2                            [Adventure, Children, Fantasy]
3                                         [Comedy, Romance]
4                                  [Comedy, Drama, Romance]
5                                                  [Comedy]
                                ...                        
161582                                       [Crime, Drama]
161594    [Action, Adventure, Animation, Drama, Fantasy,...
161918                  [Action, Adventure, Horror, Sci-Fi]
163056                 [Action, Adventure, Fantasy, Sci-Fi]
163949                                        [Documentary]
Name: genres, Length: 8737, dtype: object

# Build a content-based model
When ready, move the following class in the *models.py* script

The following script test the ContentBased class

In [5]:
def test_contentbased_class(feature_method, regressor_method):
    """Test the ContentBased class.
    Tries to make a prediction on the first (user,item ) tuple of the anti_test_set
    """
    sp_ratings = load_ratings(surprise_format=True)
    train_set = sp_ratings.build_full_trainset()

    content_algo = ContentBased(feature_method, regressor_method)
    content_algo.fit(train_set)

    anti_test_set_first = train_set.build_anti_testset()[0]

    prediction = content_algo.predict(anti_test_set_first[0], anti_test_set_first[1])

    print(prediction)

# (call here the test functions with different regressor methods)

test_contentbased_class(feature_method = None, regressor_method='random_score')

test_contentbased_class(feature_method = None,regressor_method='random_sample')

test_contentbased_class(feature_method = 'title_length',regressor_method='random_sample')

user: 277        item: 3          r_ui = None   est = 2.64   {'was_impossible': False}
user: 277        item: 3          r_ui = None   est = 3.00   {'was_impossible': False}
user: 277        item: 3          r_ui = None   est = 5.00   {'was_impossible': False}


In [ ]:
sp_ratings = load_ratings(surprise_format=True)
train_set = sp_ratings.build_full_trainset()

content_algo = ContentBased(
    features_method="all_features_V2",
    regressor_method="ridge_24"
)

content_algo.fit(train_set)

first_user = list(content_algo.user_profile.keys())[0]

explanation = content_algo.explain(first_user)

type(explanation), len(explanation)



(dict, 1324)

In [ ]:
df_explanation = pd.DataFrame(
    explanation.items(),
    columns=["feature", "importance_score"]
)

df_explanation = df_explanation.sort_values(
    by="importance_score",
    ascending=False
)

df_explanation.head(20)

,feature,importance_score
188,genome_max,1.000000
932,genome_742,0.807283
0,year,0.794843
10,decade_1990.0,0.777322
1322,movie_bayesian_mean,0.692896
909,genome_719,0.688597
1320,movie_mean_rating,0.670822
187,genome_std,0.655150
635,genome_445,0.649380
836,genome_646,0.629100


In [11]:
df_explanation["importance_score"].min(), df_explanation["importance_score"].max()

(0.0, 1.0)

The table above shows the most important features for the selected user.

A high `feature_score` means that this feature is strongly represented in the movies that the user rated highly.

Since the features are normalized, the scores are between 0 and 1.

This explanation makes the content-based recommender more interpretable because it shows which movie characteristics are the most representative of the user's preferences.